In [1]:

import pandas as pd
import re

df = pd.read_csv("student_admission_prediction.csv")

print("Dataset loaded successfully!")
print("Shape:", df.shape)

display(df.head())


c:\Users\davek\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


Dataset loaded successfully!
Shape: (500, 4)


,Entrance_Exam_Score,Undergraduate_GPA,Work_Experience,Admission_Status
0,78,7.56,Yes,0
1,91,7.51,No,0
2,68,8.99,Yes,0
3,54,8.25,Yes,0
4,82,8.51,Yes,0


In [2]:

print("Column names:")
print(df.columns.tolist())

print("\nMissing values:")
print(df.isnull().sum())

print("\nData types:")
print(df.dtypes)

print("\nWork Experience values:")
print(df["Work_Experience"].value_counts(dropna=False))

print("\nAdmission Status values:")
print(df["Admission_Status"].value_counts(dropna=False))


Column names:
['Entrance_Exam_Score', 'Undergraduate_GPA', 'Work_Experience', 'Admission_Status']

Missing values:
Entrance_Exam_Score    0
Undergraduate_GPA      0
Work_Experience        0
Admission_Status       0
dtype: int64

Data types:
Entrance_Exam_Score      int64
Undergraduate_GPA      float64
Work_Experience            str
Admission_Status         int64
dtype: object

Work Experience values:
Work_Experience
Yes    262
No     238
Name: count, dtype: int64

Admission Status values:
Admission_Status
0    497
1      3
Name: count, dtype: int64


In [ ]:

df_clean = df.copy()

def clean_experience(value):
    if pd.isna(value):
        return pd.NA

    value = str(value).strip().lower()

    if re.fullmatch(r"yes|y|true|1", value):
        return "Yes"

    elif re.fullmatch(r"no|n|false|0", value):
        return "No"

    else:
        return "INVALID"

df_clean["Work_Experience"] = (
    df_clean["Work_Experience"].apply(clean_experience)
)

print("Cleaned Work Experience values:")
print(df_clean["Work_Experience"].value_counts(dropna=False))


Cleaned Work Experience values:
Work_Experience
Yes    262
No     238
Name: count, dtype: int64


In [ ]:

numeric_columns = [
    "Entrance_Exam_Score",
    "Undergraduate_GPA"
]

for col in numeric_columns:
    original = df_clean[col].copy()

   
    text_values = original.astype("string").str.strip()

    
    valid_pattern = text_values.str.fullmatch(
        r"[+-]?\d+(?:\.\d+)?",
        na=False
    )

    
    converted = pd.to_numeric(text_values, errors="coerce")

    
    invalid = original.notna() & (
        ~valid_pattern | converted.isna()
    )

    print(f"\n{col}")
    print("Invalid values:", int(invalid.sum()))

    if invalid.any():
        print("Values requiring review:")
        print(original[invalid].head(20).tolist())

    df_clean[col] = converted



Entrance_Exam_Score
Invalid values: 0

Undergraduate_GPA
Invalid values: 0


In [ ]:

status = pd.to_numeric(
    df_clean["Admission_Status"],
    errors="coerce"
)

valid_status = status.isin([0, 1])

invalid_status = df_clean.loc[~valid_status]

print("Invalid admission status records:", len(invalid_status))
display(invalid_status.head(20))


df_clean["Admission_Status"] = status


Invalid admission status records: 0


,Entrance_Exam_Score,Undergraduate_GPA,Work_Experience,Admission_Status


In [ ]:


invalid_score = ~df_clean["Entrance_Exam_Score"].between(0, 100)
invalid_gpa = ~df_clean["Undergraduate_GPA"].between(0, 10)

print("Out-of-range exam scores:", int(invalid_score.sum()))
print("Out-of-range GPA values:", int(invalid_gpa.sum()))

print("\nInvalid exam score records:")
display(df_clean.loc[invalid_score])

print("\nInvalid GPA records:")
display(df_clean.loc[invalid_gpa])


Out-of-range exam scores: 0
Out-of-range GPA values: 0

Invalid exam score records:


,Entrance_Exam_Score,Undergraduate_GPA,Work_Experience,Admission_Status



Invalid GPA records:


,Entrance_Exam_Score,Undergraduate_GPA,Work_Experience,Admission_Status


In [7]:

quality_report = pd.DataFrame({
    "Column": df_clean.columns,
    "Missing_Values": [
        df_clean[col].isna().sum()
        for col in df_clean.columns
    ],
    "Data_Type": [
        str(df_clean[col].dtype)
        for col in df_clean.columns
    ],
    "Unique_Values": [
        df_clean[col].nunique(dropna=True)
        for col in df_clean.columns
    ]
})

print("DATA QUALITY REPORT")
display(quality_report)


DATA QUALITY REPORT


,Column,Missing_Values,Data_Type,Unique_Values
0,Entrance_Exam_Score,0,Int64,60
1,Undergraduate_GPA,0,Float64,327
2,Work_Experience,0,str,2
3,Admission_Status,0,int64,2


In [ ]:

df_final = df_clean.drop_duplicates().copy()


df_final.to_csv("admissions_regex_cleaned.csv", index=False)


quality_report.to_csv("admissions_quality_report.csv", index=False)

print("Cleaning completed!")
print("Original records:", len(df))
print("Final records:", len(df_final))
print("Duplicates removed:", len(df_clean) - len(df_final))
print("Saved: admissions_regex_cleaned.csv")
print("Saved: admissions_quality_report.csv")


Cleaning completed!
Original records: 500
Final records: 499
Duplicates removed: 1
Saved: admissions_regex_cleaned.csv
Saved: admissions_quality_report.csv
